# F3-A — Extracción de Embeddings (DistilBERT frozen)

**Objetivo**: Cargar el dataset balanceado, samplear 200k, extraer embeddings frozen de DistilBERT y engineered features. Este notebook es el **único** que extrae embeddings — los notebooks F3-B, F3-C y F3-D cargan los `.npy` generados aquí.

**Salidas en Drive**: `embeddings/train/val/test_embeddings.npy`, `embeddings/train/val/test_eng_features.npy`, `embeddings/train/val/test_labels.npy`, `embeddings/train/val/test_texts.pkl`, scaler.pkl

**Tiempo estimado**: ~40 min (GPU T4)


## 1. Instalar dependencias


In [ ]:
!pip install -q polars mlflow transformers umap-learn -U


In [ ]:
import polars as pl
import numpy as np
import torch
import gc
import os
import json
import pickle
import time
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm


## 2. Montar Google Drive

Montamos Drive para leer el parquet del EDA y persistir los embeddings.


In [ ]:
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ML/proyecto_integrador"
DATA_DIR = f"{DRIVE_BASE}/data"
PARQUET_PATH = f"{DATA_DIR}/office_products_balanced.parquet"
EMB_DIR = f"{DRIVE_BASE}/embeddings"
REPORTS_DIR = f"{DRIVE_BASE}/reports"

print(f"Parquet: {PARQUET_PATH}")
print(f"Embs:    {EMB_DIR}")
print(f"Reports: {REPORTS_DIR}")

for d in [DATA_DIR, EMB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)


## 3. Cargar datos y muestreo estratificado

Cargamos el dataset balanceado (2.5M filas, 3 clases). Sampleamos 200k estratificado manteniendo proporción entre clases. Semilla fija 42 para reproducibilidad.


In [ ]:
SAMPLE_SIZE = 200_000
BATCH_SIZE = 256
MAX_LENGTH = 128
RANDOM_STATE = 42

ENG_FEATURES = [
    'mayusculas_count', 'char_total', 'exclamacion_count',
    'interrogacion_count', 'porcentaje_mayusculas',
    'puntuacion_emocional', 'total_tokens', 'unique_types', 'ttr'
]

df = pl.read_parquet(PARQUET_PATH)
dfs = []
for s in [0, 1, 2]:
    sub = df.filter(pl.col('sentiment') == s)
    n = min(SAMPLE_SIZE // 3, sub.shape[0])
    dfs.append(sub.sample(n=n, seed=RANDOM_STATE))

df_sample = pl.concat(dfs).sample(fraction=1.0, seed=RANDOM_STATE)
print(f"Sample: {df_sample.shape}")
print(df_sample['sentiment'].value_counts().sort('sentiment'))


## 4. Train/Val/Test split

70% entrenamiento, 15% validacion, 15% prueba. Split estratificado.


In [ ]:
texts = df_sample['text'].to_list()
labels = df_sample['sentiment'].to_numpy()

X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.15, random_state=RANDOM_STATE, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=round(0.15/0.85, 3),
    random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


## 5. Extraer embeddings frozen con DistilBERT

DistilBERT en modo inference (congelado). Extraemos embedding [CLS] de 768d por reseña. Si ya existen en Drive, los carga automáticamente.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"Device: {device}")

def extract_embeddings(texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            outputs = model(**encoded)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        del encoded, outputs
        if i % (batch_size * 10) == 0:
            gc.collect()
    return np.vstack(all_embeddings)


In [ ]:
emb_paths = {
    'train': f"{EMB_DIR}/train_embeddings.npy",
    'val':   f"{EMB_DIR}/val_embeddings.npy",
    'test':  f"{EMB_DIR}/test_embeddings.npy",
}

if all(os.path.exists(p) for p in emb_paths.values()):
    print("Cargando embeddings existentes...")
    X_train_emb = np.load(emb_paths['train'])
    X_val_emb   = np.load(emb_paths['val'])
    X_test_emb  = np.load(emb_paths['test'])
    print(f"Train: {X_train_emb.shape}, Val: {X_val_emb.shape}, Test: {X_test_emb.shape}")
else:
    print("Extrayendo embeddings TRAIN...")
    X_train_emb = extract_embeddings(X_train, model, tokenizer)
    print(f"Train: {X_train_emb.shape}")
    print("Extrayendo embeddings VAL...")
    X_val_emb = extract_embeddings(X_val, model, tokenizer)
    print(f"Val: {X_val_emb.shape}")
    print("Extrayendo embeddings TEST...")
    X_test_emb = extract_embeddings(X_test, model, tokenizer)
    print(f"Test: {X_test_emb.shape}")
    for k in emb_paths:
        np.save(emb_paths[k], locals()[f'X_{k}_emb'])
    print("Embeddings guardados en Drive")


## 6. Feature engineering + scaling

Extraemos las 9 features lingüísticas del EDA y las normalizamos con Z-score. Se concatenarán con los embeddings en los notebooks de modelado.


In [ ]:
eng_paths = {
    'train': f"{EMB_DIR}/train_eng_features.npy",
    'val':   f"{EMB_DIR}/val_eng_features.npy",
    'test':  f"{EMB_DIR}/test_eng_features.npy",
}

if all(os.path.exists(p) for p in eng_paths.values()):
    print("Feature engineering ya completado, cargando...")
    eng_train = np.load(eng_paths['train'])
    eng_val   = np.load(eng_paths['val'])
    eng_test  = np.load(eng_paths['test'])
else:
    print("Computando engineered features...")
    eng_train = df_sample.filter(pl.col('text').is_in(X_train)).select(ENG_FEATURES).to_numpy()
    eng_val   = df_sample.filter(pl.col('text').is_in(X_val)).select(ENG_FEATURES).to_numpy()
    eng_test  = df_sample.filter(pl.col('text').is_in(X_test)).select(ENG_FEATURES).to_numpy()

    scaler = StandardScaler()
    eng_train = scaler.fit_transform(eng_train)
    eng_val   = scaler.transform(eng_val)
    eng_test  = scaler.transform(eng_test)

    for k in eng_paths:
        np.save(eng_paths[k], locals()[f'eng_{k}'])
    # Also save scaler
    import joblib
    joblib.dump(scaler, f"{EMB_DIR}/scaler.pkl")
    print("Engineered features guardadas en Drive")

print(f"eng_train: {eng_train.shape}, eng_val: {eng_val.shape}, eng_test: {eng_test.shape}")


## 7. Guardar metadatos (textos, labels) para notebooks downstream

Guardamos los textos crudos (necesarios para LoRA) y las etiquetas en .pkl/.npy


In [ ]:
label_paths = {
    'train': f"{EMB_DIR}/train_labels.npy",
    'val':   f"{EMB_DIR}/val_labels.npy",
    'test':  f"{EMB_DIR}/test_labels.npy",
}
for k in label_paths:
    np.save(label_paths[k], locals()[f'y_{k}'])

# Guardar textos crudos para LoRA (necesitan el texto original)
text_paths = {
    'train': f"{EMB_DIR}/train_texts.pkl",
    'val':   f"{EMB_DIR}/val_texts.pkl",
    'test':  f"{EMB_DIR}/test_texts.pkl",
}
for k in text_paths:
    with open(text_paths[k], 'wb') as f:
        pickle.dump(locals()[f'X_{k}'], f)

print("Labels y textos guardados en Drive")


## 8. Guardar embeddings para F4 (RAG)

Sample estratificado de 10k embeddings para F4.


In [ ]:
EMBEDDINGS_PATH = f"{EMB_DIR}/distilbert_embeddings_sample.npy"
LABELS_PATH = f"{EMB_DIR}/distilbert_labels_sample.npy"

emb_sample_size = 10_000
n_per_class = emb_sample_size // 3
rng = np.random.RandomState(RANDOM_STATE)
emb_chunks, label_chunks = [], []
for s in [0, 1, 2]:
    idx = np.where(y_test == s)[0]
    n = min(n_per_class, len(idx))
    chosen = rng.choice(idx, n, replace=False)
    emb_chunks.append(X_test_emb[chosen])
    label_chunks.append(y_test[chosen])

emb_sample = np.concatenate(emb_chunks)
label_sample = np.concatenate(label_chunks)

np.save(EMBEDDINGS_PATH, emb_sample)
np.save(LABELS_PATH, label_sample)
print(f"Saved {len(emb_sample)} embeddings para F4, shape: {emb_sample.shape}")


In [ ]:
# Liberar memoria
del df, df_sample, model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nF3-A completado. Ahora ejecute F3-B (clásicos), F3-C (baseline) y F3-D (LoRA+ensemble).")
